In [0]:
# ==========================================================
# 01_silver_indicador_municipio
# Camada: Silver
# Fonte: 2026-11-01_municipio.parquet (Bronze)
# Destino: default.indicador_municipio
#
# RESPONSABILIDADE
# Transformar a publicação oficial do Indicador Criança
# Alfabetizada (nível município) em tabela Silver confiável,
# com tipagem, deduplicação, rastreabilidade e Data Quality.
#
# NOTA DE ARQUITETURA
# A versão anterior deste notebook lia
# meta_alfabetizacao_municipio.parquet. Como esse arquivo
# também possui a coluna taxa_alfabetizacao, o erro passava
# despercebido e o join com a Silver de metas casava 100%.
# A fonte correta é municipio.parquet.
# ==========================================================

from pyspark.sql import functions as F

# --- Parâmetros ---
CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA_DESTINO = "default"
TABELA_DESTINO = "indicador_municipio"

BASE_BRONZE = (
    "/Workspace/Users/rm374983@fiap.com.br/"
    "postech-2iast-fase3-modelo-preditivo-g28/dados/bronze"
)
ARQUIVO_ORIGEM = "2026-11-01_municipio.parquet"
CAMINHO_ORIGEM = f"{BASE_BRONZE}/{ARQUIVO_ORIGEM}"

TABELA_FULL = f"{CATALOGO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}"

print(f"Origem : {CAMINHO_ORIGEM}")
print(f"Destino: {TABELA_FULL}")

In [0]:
# ==========================================================
# Célula 2 — Leitura da Bronze
# ==========================================================

df_raw = spark.read.parquet(CAMINHO_ORIGEM)

qtd_bruta = df_raw.count()
print(f"Registros lidos: {qtd_bruta:,}")
print(f"Colunas: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
# ==========================================================
# Célula 3 — Transformação e tipagem
#
# Decisões:
# - serie é mantida: existe no arquivo oficial e não existe
#   na tabela de metas. É parte da granularidade.
# - As 9 colunas proporcao_aluno_nivel_* são preservadas:
#   só existem no arquivo oficial e enriquecem a Gold.
# - Dedupe por (ano, id_municipio, serie, rede).
# ==========================================================

df_silver = (
    df_raw
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("id_municipio").cast("string").alias("id_municipio"),
        F.col("serie").cast("string").alias("serie"),
        F.col("rede").cast("string").alias("rede"),
        F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
        F.col("media_portugues").cast("double").alias("media_portugues"),
        *[
            F.col(f"proporcao_aluno_nivel_{n}").cast("double").alias(f"proporcao_aluno_nivel_{n}")
            for n in range(9)
        ],
    )
    .filter(F.col("ano").isNotNull() & F.col("id_municipio").isNotNull())
    .dropDuplicates(["ano", "id_municipio", "serie", "rede"])
)

print(f"Registros após transformação: {df_silver.count():,}")

In [0]:
# ==========================================================
# Célula 4 — Rastreabilidade
# Princípio da seção 3 do documento de arquitetura:
# ingested_at, source e version em todos os registros.
# ==========================================================

df_silver = (
    df_silver
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit(ARQUIVO_ORIGEM))
    .withColumn("version", F.lit("v2"))
)

df_silver.select("ano", "id_municipio", "rede", "taxa_alfabetizacao",
                 "ingested_at", "source", "version").show(5, truncate=False)

In [0]:
# ==========================================================
# Célula 5 — Data Quality
# Regras: completude, domínio e unicidade.
# ==========================================================

total = df_silver.count()

# Completude
nulos_taxa = df_silver.filter(F.col("taxa_alfabetizacao").isNull()).count()
nulos_rede = df_silver.filter(F.col("rede").isNull()).count()

# Domínio
fora_faixa = df_silver.filter(
    (F.col("taxa_alfabetizacao") < 0) | (F.col("taxa_alfabetizacao") > 100)
).count()

# Unicidade
chave = ["ano", "id_municipio", "serie", "rede"]
duplicatas = total - df_silver.dropDuplicates(chave).count()

print("=" * 55)
print("DATA QUALITY — indicador_municipio")
print("=" * 55)
print(f"Registros totais          : {total:,}")
print(f"Taxa nula                 : {nulos_taxa:,} ({nulos_taxa/total*100:.2f}%)")
print(f"Rede nula                 : {nulos_rede:,} ({nulos_rede/total*100:.2f}%)")
print(f"Taxa fora de 0-100        : {fora_faixa:,}")
print(f"Duplicatas na chave       : {duplicatas:,}")
print(f"Anos                      : {[r[0] for r in df_silver.select('ano').distinct().orderBy('ano').collect()]}")
print(f"Redes                     : {[r[0] for r in df_silver.select('rede').distinct().orderBy('rede').collect()]}")
print(f"Municípios distintos      : {df_silver.select('id_municipio').distinct().count():,}")
print("=" * 55)

# Bloqueio de promoção se houver falha crítica
assert nulos_taxa == 0, "FALHA CRÍTICA: taxa_alfabetizacao nula"
assert fora_faixa == 0, "FALHA CRÍTICA: taxa fora do domínio 0-100"
assert duplicatas == 0, "FALHA CRÍTICA: duplicatas na chave"
print("Todas as regras críticas passaram.")

In [0]:
# ==========================================================
# Célula 6 — Gravação da Silver
# ==========================================================

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_FULL))

print(f"Gravado em {TABELA_FULL}")
print(f"Registros: {spark.table(TABELA_FULL).count():,}")

In [0]:
# ==========================================================
# Célula 7 — Validação pós-gravação
# Confirma que a tabela gravada corresponde ao DataFrame.
# ==========================================================

df_check = spark.table(TABELA_FULL)

print(f"Registros na tabela : {df_check.count():,}")
print(f"Colunas             : {len(df_check.columns)}")
print(f"Anos                : {[r[0] for r in df_check.select('ano').distinct().orderBy('ano').collect()]}")
print(f"Municípios          : {df_check.select('id_municipio').distinct().count():,}")

print("\nAmostra:")
df_check.select(
    "ano", "id_municipio", "serie", "rede",
    "taxa_alfabetizacao", "media_portugues"
).show(5, truncate=False)

print("\nDistribuição por rede:")
df_check.groupBy("rede").count().orderBy("rede").show()